In [2]:
import torch, torch.nn as nn
print(torch.__version__)


2.11.0+cu130


In [3]:
torch.manual_seed(42)
X = torch.linspace(0, 1, 100)
Y = 2 * X + 1 + 0.1 * torch.randn(100)
X2 = X.unsqueeze(1)                     # (100,) -> (100, 1)
Y2 = Y.unsqueeze(1)

model = nn.Linear(1, 1)
with torch.no_grad():
    model.weight.fill_(0.0)
    model.bias.fill_(0.0)
loss_fn = nn.MSELoss()                     # mean squared error
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)    # plain gradient descent

for epoch in range(1, 501):
    optimizer.zero_grad()                    # reset the gradients
    loss = loss_fn(model(X2), Y2)
    loss.backward()
    optimizer.step()                    # update the weights
    if epoch in (1, 10, 50, 100, 200, 500):
        print(epoch, round(loss.item(), 4), round(model.weight.item(), 4), round(model.bias.item(), 4))


1 4.3716 0.2345 0.4012
10 0.1179 0.9612 1.451
50 0.0389 1.4192 1.317
100 0.0172 1.7014 1.1658
200 0.0101 1.9182 1.0496
500 0.0096 1.9928 1.0096


- The optimizer call optimizer.step() replaced the manual weight update step w -= lr * w.grad, while optimizer.zero_grad() replaced the gradient-clearing call w.grad.zero_().

In [5]:
#Cell E8: your own model class
class LinearModel(nn.Module):
  def __init__(self):
    super().__init__()
    self.fc = nn.Linear(1,1)

  def forward(self, x):
    return self.fc(x)

m = LinearModel()
with torch.no_grad():
  m.fc.weight.fill_(0.0); m.fc.bias.fill_(0.0)
opt = torch.optim.SGD(m.parameters(), lr=0.1)
for epoch in range(500):
  opt.zero_grad(); loss = loss_fn(m(X2), Y2); loss.backward(); opt.step()
print(round(loss.item(), 4), round(m.fc.weight.item(), 4), round(m.fc.bias.item(), 4))
print(m)

0.0096 1.9928 1.0096
LinearModel(
  (fc): Linear(in_features=1, out_features=1, bias=True)
)


In [8]:
#Cell E9: data that a straight line cannot fit
torch.manual_seed(0)
Xn = torch.linspace(0, 1, 200).unsqueeze(1)
Yn = 4 * (Xn - 0.5) ** 2 + 0.05 * torch.randn(200, 1)
print("variance of Y:", round(Yn.var(unbiased=False).item(), 4))

def train(model, opt, epochs, show):
  for e in range(1, epochs + 1):
    opt.zero_grad()
    loss = loss_fn(model(Xn), Yn)
    loss.backward()
    opt.step()
    if e in show:
      print(e, round(loss.item(), 4))

torch.manual_seed(2)
lin = nn.Linear(1,1)
train(lin, torch.optim.SGD(lin.parameters(), lr=0.1), 3000, (1, 100, 500, 3000))

variance of Y: 0.0902
1 0.3057
100 0.0913
500 0.0901
3000 0.0901


- The straight line end no better than predicting thhe mean because a straight line can only slope up or down, but the parabola is symmetric, so the left and right sides cancel and the best line is flat.

In [10]:
#Cell E10: add a hidden layer with ReLU
torch.manual_seed(2)
mlp = nn.Sequential(nn.Linear(1,16), nn.ReLU(), nn.Linear(16,1))
train(mlp, torch.optim.Adam(mlp.parameters(), lr = 0.01), 3000, (1, 100, 500, 1000, 3000))
print(sum(p.numel() for p in mlp.parameters()), "parameters")
with torch.no_grad():
  print(mlp(torch.tensor([[0.0], [0.5], [1.0]])).flatten())

1 0.286
100 0.0408
500 0.003
1000 0.0029
3000 0.0025
49 parameters
tensor([ 0.9512, -0.0152,  0.9255])


- The network learn to capture coomplex bending curves and non-linear patterns that a rigid straight line is physically incapable of representing.Uses activation functions to break that straight-line rule. Every time data passes through a ReLU function.

In [12]:
#Cell E11: take the ReLU out
torch.manual_seed(2)
no_act = nn.Sequential(nn.Linear(1, 16), nn.Linear(16, 1))
train(no_act, torch.optim.Adam(no_act.parameters(), lr=0.01), 3000, (1, 100, 3000))

1 0.7118
100 0.0901
3000 0.0901


- It has 16 hidden units and still cannot fit the curve line because there is no non-linear activation function between the layers.

In [13]:
#Cell E12 (bonus): an unlucky start
torch.manual_seed(1)
mlp = nn.Sequential(nn.Linear(1,16), nn.ReLU(), nn.Linear(16,1))
train(mlp, torch.optim.Adam(mlp.parameters(), lr = 0.01), 3000, (1, 100, 500, 1000, 3000))
print(sum(p.numel() for p in mlp.parameters()), "parameters")
with torch.no_grad():
  print(mlp(torch.tensor([[0.0], [0.5], [1.0]])).flatten())

1 0.2255
100 0.0901
500 0.0901
1000 0.0901
3000 0.0901
49 parameters
tensor([0.3436, 0.3356, 0.3275])


- The model suffered, where the random starting weights pushed the hidden neurons into the negative zone, causing the ReLU function to output a flat zero with zero gradient slope across all training samples, which permanently locked the weights and blocked the optimizer from updating them.